# Credit Card Default: Sensitive Balancing Preprocessing

Author: Ilse Harmers \
Last modified: February 24, 2026

In [ ]:
# Importing libraries.
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import utils

## Preprocessing Credit Card Default Dataset

In [ ]:
# Importing Credit Card Default dataset (downloaded from https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients on May 28, 2025).

# Assigning file path for importing the dataset. 
path = "../credit-card-default/credit-card-clients.xls"

# Importing dataset as pandas DataFrame.
credit = pd.read_excel(path, index_col = 0, header = 1)
credit = credit.rename(columns = {"default payment next month": "DEFAULT"})
dataset_name = "Credit"
column_names = list(credit.columns)
credit

In [ ]:
# Print names of columns in the dataset as a check.
print(f"{dataset_name} contains {len(column_names)} columns: \n {column_names} \n")

# Check dtypes of dataset's columns.
print("Columns' dtypes:\n", credit.dtypes)
print()

# Count missing values (reported as NaNs) in each column.
missing_values = np.array([credit[col].isna().sum() for col in column_names])

# This if-elif statement checks whether the columns in the dataset contain missing values that are reported as NaNs. 
if missing_values.any() == False:   # no non-zero entries
    print(f"{dataset_name} has no missing values in any of its columns.")
elif missing_values.any() == True:   # at least one non-zero entry (indicating the presence of missing values)
    print(f"There are {np.sum(missing_values)} missing values in {dataset_name}'s columns.")   
    for i in range(len(missing_values)):
        print(f"\t{column_names[i]}: {missing_values[i]} values")

In [ ]:
print("Number of rows with unusual labels: ", 
      credit.loc[(credit["EDUCATION"] == 0) | (credit["EDUCATION"] == 5) | (credit["EDUCATION"] == 6) | (credit["MARRIAGE"] == 0)].shape[0])

In [ ]:
credit = credit.drop(index=credit.loc[(credit["EDUCATION"] == 0) | (credit["EDUCATION"] == 5) | 
                     (credit["EDUCATION"] == 6) | (credit["MARRIAGE"] == 0)].index)

In [ ]:
# Statistical summary of the dataset.
credit.describe()

## Sensitive Balancing Preprocessing

In [ ]:
# Splitting the Credit dataset into a train and test set with a 80:20 ratio.
credit_train, credit_test = train_test_split(credit, train_size = 0.80, random_state = 42)

In [ ]:
def sens_preprocessing(df):
    """This function carries out the sensitive balancing preprocessing by randomly undersampling the demographic groups such that each group 
    has the same size. The demographic groups for Credit are Male-0 (M0), Male-1 (M1), Female-0 (F0) and Female-1 (F1). For Credit, M1 is the smallest 
    group. So, M0, F0 and F1 will be undersampled in order to have the same size as M1."""
    
    females = df[df["SEX"] == 2].shape[0]
    print("females: ", females) 
    males = df[df["SEX"] == 1].shape[0]
    print("males: ", males) 

    # Creating a new column in the DataFrame with the demographic group labels.
    df.loc[df.loc[(df["SEX"] == 1) & (df["DEFAULT"] == 0)].index, "sex-default"] = "M0"
    df.loc[df.loc[(df["SEX"] == 1) & (df["DEFAULT"] == 1)].index, "sex-default"] = "M1"
    df.loc[df.loc[(df["SEX"] == 2) & (df["DEFAULT"] == 0)].index, "sex-default"] = "F0"
    df.loc[df.loc[(df["SEX"] == 2) & (df["DEFAULT"] == 1)].index, "sex-default"] = "F1"
    m1_rows = df[df["sex-default"] == "M1"].shape[0]
    print("Rows of M1: ", m1_rows)
    f1_rows = df[df["sex-default"] == "F1"].shape[0]
    print("Rows of F1: ", f1_rows)

    # Plotting the label distribution of the demographic groups before preprocessing.
    utils.countplot(data = df, column_name = "sex-default", order = ["F0", "F1", "M0", "M1"], fig_size = (8, 5))

    # Undersampling the demographic groups such that all demographic groups have the same size.
    df_bal = pd.concat([df[df["sex-default"] == "M0"].sample(m1_rows, random_state = 42),
                        df[df["sex-default"] == "M1"],
                        df[df["sex-default"] == "F0"].sample(m1_rows, random_state = 42),
                        df[df["sex-default"] == "F1"].sample(m1_rows, random_state = 42)], axis = 0)

    # Plotting the label distribution of the demographic groups after preprocessing.
    utils.countplot(data = df_bal, column_name = "sex-default", order = ["F0", "F1", "M0", "M1"], fig_size = (6, 5))

    # Resetting the DataFrame.
    df_bal = df_bal.drop(columns = ["sex-default"])
    df = df.drop(columns = ["sex-default"])

    return df_bal

In [ ]:
# Preprocessing the train and test set.
train_bal = sens_preprocessing(credit_train)
test_bal = sens_preprocessing(credit_test)

In [ ]:
# Statistical summary of the train set.
print(train_bal.describe())

# Saving train and test splits to csv files.
#test_bal.to_csv("./train-test-datasets/credit-card-default/credit_test.csv", index = False)
#train_bal.to_csv("./train-test-datasets/credit-card-default/credit_train.csv", index = False)

In [ ]:
# Computing the demographic parity and disparate impact metrics for the train and test 'balanced' datasets.
df = train_bal #test_bal
# Encoding the sensitive and target attributes for the fairness analysis.
sex_encoded = utils.one_hot_encode(df[["SEX"]], order = [[2, 1]])
train_encoded = pd.concat([sex_encoded.reset_index(drop = True), df["DEFAULT"].reset_index(drop = True)], axis = 1)

print("Demographic parity: ", utils.demographic_parity(df = train_encoded, s = "SEX", y = "DEFAULT"))
print("Disparate impact: ", utils.disparate_impact(df = train_encoded, s = "SEX", y = "DEFAULT"))